# AirWindow BD — PM2.5 / PM10 forecasting (1, 3, 6 h)

Training data: **CAMS-BD** hourly observations from Bangladesh DoE Continuous Air Monitoring Stations
(doi:10.17632/hwmzyh5w4w.1, CC BY 4.0), cleaned by `airwindow.build_dataset` (see `DATA_QUALITY.md`
in the attached dataset), plus **Open-Meteo ERA5** hourly weather at each station.

* Chronological split on issue time: train → targets ≤ 2023-01-01, validation = 2023, **test = 2024 (untouched)**.
* Models: persistence, seasonal-naive (same hour yesterday), LightGBM (point + q10/q90 with conformal
  calibration), GRU (GPU). Each with weather known at issue time, and with weather at the target hour
  (ERA5 as a stand-in for the forecast the app fetches — optimistic, reported separately).
* Leave-one-station-out evaluation for station generalisation.

Attach the dataset containing `cams_bd_hourly.parquet` and `era5_station_weather.parquet`, enable a GPU, then Run All.


In [ ]:
!mkdir -p /kaggle/working/src/airwindow

In [ ]:
%%writefile /kaggle/working/src/airwindow/__init__.py
"""AirWindow BD data and forecasting package."""


In [ ]:
%%writefile /kaggle/working/src/airwindow/parsing.py
"""CAMS-BD workbook parsing.

The four CAMS-BD workbooks do not share a schema. Known problems handled here
(each one is counted in the returned audit so the data-quality report can show it):

* 2012-2021: one sheet of stacked station blocks. Every block has its own header
  row and unit row. The Savar block's header swaps "Solar Rad" and "BP", so the
  columns are mapped per block from that block's own header, not the sheet header.
* 2012-2021: dates were originally dd/mm/yyyy text. Excel converted those with
  day <= 12 into serial dates with day and month swapped and left the rest as
  text. Some ranges are correct serials, some carry the hour as a fraction.
  We decode each row into candidate timestamps and pick the one that continues
  the station's hourly sequence (see ``_decode_sequence``).
* 2012-2021: the Khulna, Rajshahi and Sangsad blocks contain no dates at all,
  only an hour-of-day. Their timestamps cannot be verified, so those blocks are
  excluded rather than reconstructed from row position.
* 2022-2024: a single "Date & Time" column, PM10/PM2.5 column order differs by
  year, and some stations' meteorology columns are shifted (handled later by
  physical-range validation in ``cleaning.py``).

Timestamps: the source uses hour-ending labels 01:00..24:00 in Bangladesh local
time. ``ts`` is the end of the averaging hour as naive Asia/Dhaka time (UTC+6,
no DST); 24:00 becomes 00:00 of the next day.
"""
from __future__ import annotations

import datetime as dt
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd

CANONICAL = ["so2", "no", "no2", "nox", "co", "o3", "pm25", "pm10",
             "ws", "wd", "temp", "rh", "sr", "bp", "rain", "vws"]

# Source header label (lower-cased, stripped) -> canonical column.
HEADER_MAP = {
    "so2": "so2", "no": "no", "no2": "no2", "nox": "nox", "co": "co", "o3": "o3",
    "pm2.5": "pm25", "pm10": "pm10",
    "wind speed": "ws", "ws": "ws", "wind dir": "wd", "wd": "wd",
    "temperature": "temp", "temp": "temp", "rh": "rh",
    "solar rad": "sr", "sr": "sr", "bp": "bp", "rain": "rain",
    "v wind speed": "vws", "vws": "vws",
}

# Expected unit per canonical column; anything else is reported, not converted.
EXPECTED_UNITS = {
    "so2": {"ppb"}, "no": {"ppb"}, "no2": {"ppb"}, "nox": {"ppb"}, "co": {"ppm"},
    "o3": {"ppb"}, "pm25": {"ug/m3"}, "pm10": {"ug/m3"}, "ws": {"m/s"},
    "wd": {"deg", "degre-----", "degree----"}, "temp": {"c°", "degreec"},
    "rh": {"%"}, "sr": {"w/m2"}, "bp": {"mb", "hpa"}, "rain": {"mm", "m m"},
    "vws": {"m/s"},
}

# Raw station label -> (station_id, display name). CDA is taken to be the
# Agrabad (Chattogram) site; this is an assumption recorded in the data card.
STATION_MAP = {
    "agrabad chittagong": "ctg_agrabad", "cda": "ctg_agrabad",
    "tv st-chittagong": "ctg_tv", "tv center": "ctg_tv", "tv sation": "ctg_tv",
    "barc": "dhk_barc", "darussalam": "dhk_darussalam", "sangsad": "dhk_sangsad",
    "doe": "dhk_doe", "gazipur": "gazipur", "narayonganj": "narayanganj",
    "narayangonj": "narayanganj", "narayanganj": "narayanganj",
    "savar": "savar", "narsingdi": "narsingdi", "mymensingh": "mymensingh",
    "mymensing": "mymensingh", "cumilla": "cumilla", "rajshahi": "rajshahi",
    "khulna": "khulna", "barishal": "barishal", "sylhet": "sylhet",
    "rangpur": "rangpur",
}

EXCEL_EPOCH = pd.Timestamp("1899-12-30")
_TIME_RE = re.compile(r"(\d{1,2}):(\d{2})(?::\d{2})?\s*([AP]M)?\s*$", re.I)


@dataclass
class Audit:
    rows_in: int = 0
    rows_out: int = 0
    dropped: Dict[str, int] = field(default_factory=dict)
    notes: List[str] = field(default_factory=list)
    unit_issues: List[dict] = field(default_factory=list)
    date_fixes: Dict[str, int] = field(default_factory=dict)

    def drop(self, reason: str, n: int) -> None:
        if n:
            self.dropped[reason] = self.dropped.get(reason, 0) + int(n)

    def to_dict(self) -> dict:
        return {"rows_in": self.rows_in, "rows_out": self.rows_out,
                "dropped": self.dropped, "notes": self.notes,
                "unit_issues": self.unit_issues, "date_fixes": self.date_fixes}


def station_id(raw: str) -> Optional[str]:
    return STATION_MAP.get(str(raw).strip().lower())


def parse_hour(x) -> Optional[int]:
    """Hour-ending label -> 1..24. Accepts '01:00', '24:00:00 PM', 8, time objects."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    if isinstance(x, dt.time):
        return x.hour or 24
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x == int(x)):
        h = int(x)
        return h if 0 <= h <= 24 else None
    m = _TIME_RE.search(str(x))
    if not m:
        return None
    h = int(m.group(1))
    ampm = (m.group(3) or "").upper()
    if h != 24 and ampm == "PM" and h < 12:
        h += 12
    elif ampm == "AM" and h == 12:
        h = 0
    if h == 0:
        h = 24
    return h if 1 <= h <= 24 else None


def _swap(d: pd.Timestamp) -> Optional[pd.Timestamp]:
    if d.day > 12 or d.day == d.month:
        return None
    try:
        return d.replace(month=d.day, day=d.month)
    except ValueError:
        return None


def date_candidates(x, hour: Optional[int]) -> List[pd.Timestamp]:
    """All plausible end-of-hour timestamps for one 2012-2021 row, as-is reading first."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, (int, float, np.integer, np.floating)):
        # Round to the hour first: float error turns 00:00 into 23:59:59.99.
        serial = round(float(x) * 24) / 24
        day = EXCEL_EPOCH + pd.Timedelta(days=int(np.floor(serial)))
        h_frac = int(round((serial - np.floor(serial)) * 24))
        if hour is None:
            if h_frac == 0:
                # A bare midnight serial is 24:00 of the previous day.
                return [day]
            hour = h_frac
        serial_ts = day + pd.Timedelta(hours=h_frac)
        out = []
        if h_frac == 0 and hour == 24:
            # Midnight serial labelled 24:00 is that same instant.
            out.append(serial_ts)
        bases = [day] + ([s] if (s := _swap(day)) is not None else [])
        out += [b + pd.Timedelta(hours=hour) for b in bases]
        if h_frac:
            # Some ranges store the full timestamp, or timestamp - 1h, in the serial.
            out += [serial_ts, serial_ts + pd.Timedelta(hours=1)]
        return list(dict.fromkeys(out))
    if isinstance(x, (dt.datetime, pd.Timestamp)):
        d = pd.Timestamp(x)
        if hour is None:
            return [d]
        return [d.normalize() + pd.Timedelta(hours=hour)]
    s = str(x).strip()
    out = []
    for fmt in ("%d/%m/%Y", "%m/%d/%Y"):
        try:
            d = pd.to_datetime(s, format=fmt)
        except (ValueError, TypeError):
            continue
        if hour is not None:
            d = d + pd.Timedelta(hours=hour)
        if d not in out:
            out.append(d)
    return out


def _decode_sequence(cands: List[List[pd.Timestamp]], audit: Audit, max_gap_days: int = 400):
    """Choose one timestamp per row so the station series runs forward hourly.

    Preference order: the candidate equal to previous+1h; else the earliest
    candidate after the previous timestamp within ``max_gap_days``. Rows with no
    candidate that moves forward are dropped (counted as out-of-sequence).
    """
    one_h = pd.Timedelta(hours=1)
    max_gap = pd.Timedelta(days=max_gap_days)
    chosen: List[Optional[pd.Timestamp]] = []
    prev: Optional[pd.Timestamp] = None
    for c in cands:
        if not c:
            chosen.append(None)
            continue
        if prev is None:
            pick = c[0]
        elif prev + one_h in c:
            pick = prev + one_h
        else:
            fwd = [t for t in c if prev < t <= prev + max_gap]
            if not fwd:
                audit.date_fixes["out_of_sequence_dropped"] = audit.date_fixes.get("out_of_sequence_dropped", 0) + 1
                chosen.append(None)
                continue
            pick = min(fwd)
            audit.date_fixes["sequence_gaps"] = audit.date_fixes.get("sequence_gaps", 0) + 1
        if pick != c[0]:
            audit.date_fixes["day_month_swapped"] = audit.date_fixes.get("day_month_swapped", 0) + 1
        chosen.append(pick)
        prev = pick
    return chosen


def _check_units(station: str, header: List[str], units: List, audit: Audit, source: str) -> None:
    for h, u in zip(header, units):
        col = HEADER_MAP.get(str(h).strip().lower())
        if col is None or u is None or (isinstance(u, float) and np.isnan(u)):
            continue
        u_norm = str(u).strip().lower()
        if u_norm not in EXPECTED_UNITS[col]:
            audit.unit_issues.append({"source": source, "station": station, "column": col, "unit": str(u).strip()})


def parse_2012_2021(path: Path, audit: Optional[Audit] = None) -> pd.DataFrame:
    audit = audit or Audit()
    raw = pd.read_excel(path, header=None, dtype=object) if not isinstance(path, pd.DataFrame) else path
    audit.rows_in += len(raw) - 1
    # A block header is any row whose Time cell reads "Time".
    hdr_rows = raw.index[raw[2].astype(str).str.strip() == "Time"].tolist()
    if not hdr_rows or hdr_rows[0] != 0:
        hdr_rows = [0] + hdr_rows
    bounds = list(zip(hdr_rows, hdr_rows[1:] + [len(raw)]))
    frames = []
    for start, end in bounds:
        header = [str(v).strip() if v is not None else "" for v in raw.iloc[start]]
        block = raw.iloc[start + 1:end]
        units_mask = block[3].astype(str).str.strip().str.lower() == "ppb"
        locs = block[0].dropna().unique()
        # The sheet's first header row also heads the first station's block.
        for loc in locs:
            g = block[block[0] == loc]
            sid = station_id(loc)
            urows = g[units_mask.reindex(g.index, fill_value=False)]
            if len(urows):
                _check_units(sid, header, urows.iloc[0].tolist(), audit, "2012-2021")
            g = g[~units_mask.reindex(g.index, fill_value=False)]
            if sid is None:
                audit.drop("unknown_station", len(g))
                continue
            has_date = g[1].notna() & (g[1].astype(str).str.strip() != "")
            if not has_date.any():
                audit.drop(f"no_date_column_{sid}", len(g))
                audit.notes.append(f"2012-2021 {loc}: block has no dates ({len(g)} rows); excluded, not reconstructed.")
                continue
            hours = [parse_hour(x) for x in g[2]]
            cands = [date_candidates(d, h) for d, h in zip(g[1], hours)]
            ts = _decode_sequence(cands, audit)
            out = pd.DataFrame({"station_id": sid, "ts": ts}, index=g.index)
            for j, h in enumerate(header):
                col = HEADER_MAP.get(h.lower())
                if col:
                    out[col] = pd.to_numeric(g[j], errors="coerce")
            n_bad = out["ts"].isna().sum()
            audit.drop("unparseable_timestamp", n_bad)
            frames.append(out.dropna(subset=["ts"]))
    df = pd.concat(frames, ignore_index=True)
    df["source_file"] = "2012-2021"
    return df


def parse_annual(path, year_label: str, audit: Optional[Audit] = None) -> pd.DataFrame:
    """2022, 2023 and 2024 workbooks: one 'Date & Time' column, header in row 0."""
    audit = audit or Audit()
    raw = pd.read_excel(path, header=None, dtype=object) if not isinstance(path, pd.DataFrame) else path
    header = [str(v).strip() if v is not None else "" for v in raw.iloc[0]]
    body = raw.iloc[1:]
    audit.rows_in += len(body)
    units_mask = body[2].astype(str).str.strip().str.lower() == "ppb"
    for loc, u in body[units_mask].groupby(0):
        _check_units(station_id(loc), header, u.iloc[0].tolist(), audit, year_label)
    body = body[~units_mask]
    audit.drop("unit_rows", int(units_mask.sum()))
    ts = pd.to_datetime(body[1], errors="coerce")
    sids = body[0].map(station_id)
    out = pd.DataFrame({"station_id": sids, "ts": ts})
    for j, h in enumerate(header):
        col = HEADER_MAP.get(h.lower())
        if col:
            out[col] = pd.to_numeric(body[j], errors="coerce")
    audit.drop("unknown_station", int(out["station_id"].isna().sum()))
    out = out.dropna(subset=["station_id"])
    bad = out["ts"].isna()
    audit.drop("unparseable_timestamp", int(bad.sum()))
    out = out[~bad].copy()
    # Snap to the hour; sub-hour stamps are reported rather than silently kept.
    off = out["ts"] != out["ts"].dt.round("h")
    audit.drop("non_hourly_timestamp", int(off.sum()))
    out = out[~off]
    out["source_file"] = year_label
    return out.reset_index(drop=True)


def parse_all(raw_dir: Path) -> tuple[pd.DataFrame, Audit]:
    raw_dir = Path(raw_dir)
    audit = Audit()
    parts = [parse_2012_2021(raw_dir / "Bangladesh_CAMS_AirQuality_Meteorology_2012-2021.xlsx", audit)]
    for y in ("2022", "2023", "2024"):
        parts.append(parse_annual(raw_dir / f"Bangladesh_CAMS_AirQuality_Meteorology_{y}.xlsx", y, audit))
    df = pd.concat(parts, ignore_index=True)
    for c in CANONICAL:
        if c not in df:
            df[c] = np.nan
    return df[["station_id", "ts", *CANONICAL, "source_file"]], audit


In [ ]:
%%writefile /kaggle/working/src/airwindow/cleaning.py
"""Validation and cleaning after parsing. Never fills values; only nulls bad ones.

Rules (all counted into the quality report):
1. Physical range per column. Out-of-range values become NaN.
2. Station-year quarantine: if more than QUARANTINE_FRAC of a column's
   non-null values in one station-year fail the range check, the column is
   treated as shifted/mislabelled for that station-year and nulled entirely.
3. Stuck sensor: PM values repeated unchanged for >= STUCK_HOURS consecutive
   hours are nulled (a flat line is an instrument fault, not air).
4. Duplicates on (station_id, ts): identical rows collapse; conflicting rows
   keep the later source file and the conflict count is reported.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from .parsing import CANONICAL

RANGES = {
    "pm25": (0.5, 1000.0), "pm10": (0.5, 2000.0),
    "so2": (0, 500), "no": (0, 1000), "no2": (0, 500), "nox": (0, 1500),
    "co": (0, 50), "o3": (0, 300),
    "ws": (0, 40), "wd": (0, 360), "temp": (0, 48), "rh": (1, 100),
    "sr": (0, 1400), "bp": (950, 1050), "rain": (0, 200), "vws": (-10, 10),
}
QUARANTINE_FRAC = 0.2
STUCK_HOURS = 6


def _stuck_mask(s: pd.Series, n: int) -> pd.Series:
    run_id = (s != s.shift()).cumsum()
    run_len = s.groupby(run_id).transform("size")
    return s.notna() & (run_len >= n)


def clean(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    report: dict = {"range_nulled": {}, "quarantined": [], "stuck_nulled": {}, "duplicates": {}}
    df = df.copy()
    # Hour-ending stamps: 00:00 on 1 Jan closes the previous year.
    df["year"] = (df["ts"] - pd.Timedelta(hours=1)).dt.year

    # 2. quarantine first, so whole shifted columns are reported as such.
    for col, (lo, hi) in RANGES.items():
        bad = df[col].notna() & ~df[col].between(lo, hi)
        frac = bad.groupby([df["station_id"], df["year"]]).sum() / \
            df[col].notna().groupby([df["station_id"], df["year"]]).sum().replace(0, np.nan)
        for (sid, yr), f in frac[frac > QUARANTINE_FRAC].items():
            m = (df["station_id"] == sid) & (df["year"] == yr)
            report["quarantined"].append({"station_id": sid, "year": int(yr), "column": col,
                                          "bad_frac": round(float(f), 3),
                                          "n_nulled": int(df.loc[m, col].notna().sum())})
            df.loc[m, col] = np.nan
        # 1. remaining isolated out-of-range values.
        bad = df[col].notna() & ~df[col].between(lo, hi)
        report["range_nulled"][col] = int(bad.sum())
        df.loc[bad, col] = np.nan

    # 4. duplicates
    key = ["station_id", "ts"]
    dup_any = df.duplicated(key, keep=False)
    exact = df.duplicated(key + CANONICAL, keep="first")
    df = df[~exact]
    conflicts = df.duplicated(key, keep=False)
    report["duplicates"] = {"rows_in_duplicate_keys": int(dup_any.sum()),
                            "exact_dropped": int(exact.sum()),
                            "conflicting_keys": int(df[conflicts].drop_duplicates(key).shape[0])}
    order = {"2012-2021": 0, "2022": 1, "2023": 2, "2024": 3}
    df = (df.assign(_o=df["source_file"].map(order)).sort_values(key + ["_o"])
            .drop_duplicates(key, keep="last").drop(columns="_o"))

    # 3. stuck sensors, per station in time order
    df = df.sort_values(key).reset_index(drop=True)
    for col in ("pm25", "pm10"):
        m = df.groupby("station_id", group_keys=False)[col].apply(lambda s: _stuck_mask(s, STUCK_HOURS))
        report["stuck_nulled"][col] = int(m.sum())
        df.loc[m, col] = np.nan

    return df.drop(columns="year"), report


def to_hourly_grid(df: pd.DataFrame) -> pd.DataFrame:
    """Reindex each station onto a complete hourly index (missing hours = NaN rows)."""
    out = []
    for sid, g in df.groupby("station_id"):
        idx = pd.date_range(g["ts"].min(), g["ts"].max(), freq="h", name="ts")
        g = g.set_index("ts").reindex(idx)
        g["station_id"] = sid
        out.append(g.reset_index())
    return pd.concat(out, ignore_index=True)


In [ ]:
%%writefile /kaggle/working/src/airwindow/stations.py
"""CAMS station metadata.

Coordinates are approximate site locations (city/landmark level, roughly
within 2 km). They are good enough for matching to a ~10 km weather grid and
for showing a pin on a map; they are not surveyed sensor positions.
"""
STATIONS = {
    "dhk_barc":       {"name": "Dhaka – BARC, Farmgate", "name_bn": "ঢাকা – বিএআরসি, ফার্মগেট", "city": "Dhaka", "lat": 23.759, "lon": 90.389},
    "dhk_darussalam": {"name": "Dhaka – Darussalam", "name_bn": "ঢাকা – দারুস সালাম", "city": "Dhaka", "lat": 23.781, "lon": 90.356},
    "dhk_doe":        {"name": "Dhaka – DoE, Agargaon", "name_bn": "ঢাকা – পরিবেশ অধিদপ্তর, আগারগাঁও", "city": "Dhaka", "lat": 23.778, "lon": 90.379},
    "dhk_sangsad":    {"name": "Dhaka – Sangsad Bhaban", "name_bn": "ঢাকা – সংসদ ভবন", "city": "Dhaka", "lat": 23.762, "lon": 90.378},
    "gazipur":        {"name": "Gazipur", "name_bn": "গাজীপুর", "city": "Gazipur", "lat": 23.994, "lon": 90.420},
    "narayanganj":    {"name": "Narayanganj", "name_bn": "নারায়ণগঞ্জ", "city": "Narayanganj", "lat": 23.631, "lon": 90.508},
    "savar":          {"name": "Savar", "name_bn": "সাভার", "city": "Savar", "lat": 23.858, "lon": 90.267},
    "narsingdi":      {"name": "Narsingdi", "name_bn": "নরসিংদী", "city": "Narsingdi", "lat": 23.921, "lon": 90.717},
    "ctg_agrabad":    {"name": "Chattogram – Agrabad", "name_bn": "চট্টগ্রাম – আগ্রাবাদ", "city": "Chattogram", "lat": 22.323, "lon": 91.812},
    "ctg_tv":         {"name": "Chattogram – TV Station, Khulshi", "name_bn": "চট্টগ্রাম – টিভি স্টেশন, খুলশী", "city": "Chattogram", "lat": 22.356, "lon": 91.807},
    "rajshahi":       {"name": "Rajshahi", "name_bn": "রাজশাহী", "city": "Rajshahi", "lat": 24.369, "lon": 88.625},
    "khulna":         {"name": "Khulna", "name_bn": "খুলনা", "city": "Khulna", "lat": 22.816, "lon": 89.565},
    "sylhet":         {"name": "Sylhet", "name_bn": "সিলেট", "city": "Sylhet", "lat": 24.889, "lon": 91.867},
    "barishal":       {"name": "Barishal", "name_bn": "বরিশাল", "city": "Barishal", "lat": 22.700, "lon": 90.365},
    "mymensingh":     {"name": "Mymensingh", "name_bn": "ময়মনসিংহ", "city": "Mymensingh", "lat": 24.757, "lon": 90.408},
    "cumilla":        {"name": "Cumilla", "name_bn": "কুমিল্লা", "city": "Cumilla", "lat": 23.461, "lon": 91.180},
    "rangpur":        {"name": "Rangpur", "name_bn": "রংপুর", "city": "Rangpur", "lat": 25.746, "lon": 89.250},
}


In [ ]:
%%writefile /kaggle/working/src/airwindow/weather.py
"""Open-Meteo weather at each station (no API key needed).

Training uses the ERA5 archive; the live app uses the forecast API with the
same variable names, so the model sees the same kind of input in both places.
Open-Meteo labels hours by their start in local time; precipitation is the sum
over the preceding hour. We key both on the same local clock hour as CAMS ``ts``.
"""
from __future__ import annotations

import json
import time
import urllib.parse
import urllib.request

import pandas as pd

VARS = ["temperature_2m", "relative_humidity_2m", "wind_speed_10m", "wind_direction_10m",
        "precipitation", "surface_pressure", "boundary_layer_height"]
ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
FORECAST_URL = "https://api.open-meteo.com/v1/forecast"
RENAME = {"temperature_2m": "w_temp", "relative_humidity_2m": "w_rh", "wind_speed_10m": "w_ws",
          "wind_direction_10m": "w_wd", "precipitation": "w_rain", "surface_pressure": "w_bp",
          "boundary_layer_height": "w_blh"}


def _get(url: str, params: dict, retries: int = 6) -> dict:
    q = url + "?" + urllib.parse.urlencode(params)
    for i in range(retries):
        try:
            with urllib.request.urlopen(q, timeout=60) as r:
                return json.load(r)
        except Exception as e:
            if i == retries - 1:
                raise
            # 429 = minute/hour quota; back off long enough for the window to reset.
            time.sleep(70 if getattr(e, "code", None) == 429 else 5 * (i + 1))
    raise RuntimeError("unreachable")


def _frame(payload: dict) -> pd.DataFrame:
    h = payload["hourly"]
    df = pd.DataFrame({k: h[k] for k in VARS})
    df.insert(0, "ts", pd.to_datetime(h["time"]))
    return df.rename(columns=RENAME)


def fetch_archive(lat: float, lon: float, start: str, end: str) -> pd.DataFrame:
    return _frame(_get(ARCHIVE_URL, {"latitude": lat, "longitude": lon, "start_date": start,
                                     "end_date": end, "hourly": ",".join(VARS),
                                     "timezone": "Asia/Dhaka", "wind_speed_unit": "ms"}))


def fetch_forecast(lat: float, lon: float, past_days: int = 2, forecast_days: int = 2) -> pd.DataFrame:
    return _frame(_get(FORECAST_URL, {"latitude": lat, "longitude": lon, "hourly": ",".join(VARS),
                                      "past_days": past_days, "forecast_days": forecast_days,
                                      "timezone": "Asia/Dhaka", "wind_speed_unit": "ms"}))


In [ ]:
%%writefile /kaggle/working/src/airwindow/features.py
"""Forecast features. Every feature uses only what is known at issue time t.

Input is one station on a complete hourly grid (missing hours present as NaN).
Each output row is an issue time t with targets y_{pol}_h{h} = value at t+h.

Weather comes in two forms:
* ``w_*_now``  - weather at t (observed/analysis; always available at issue time)
* ``w_*_tgt``  - weather at t+h. In training this is ERA5 reanalysis, standing in
  for the weather *forecast* the live app fetches; it is therefore an optimistic
  proxy and models using it are reported separately ("with weather forecast").
Rolling statistics use ``min_periods`` and only past rows; nothing is filled
from the future. No global scaling is fitted here (trees need none; the neural
model fits its scaler on the training split only).
"""
from __future__ import annotations

import numpy as np
import pandas as pd

POLLUTANTS = ("pm25", "pm10")
HORIZONS = (1, 3, 6)
LAGS = (1, 2, 3, 6, 12, 23)  # lag 0 is the value at t itself
WINDOWS = (3, 6, 24)
WEATHER = ("w_temp", "w_rh", "w_ws", "w_wd", "w_rain", "w_bp", "w_blh")
MAX_OBS_AGE_H = 3  # issue a forecast only if the latest PM reading is this fresh


def _hours_since_valid(s: pd.Series) -> pd.Series:
    idx = np.arange(len(s), dtype=float)
    last = pd.Series(np.where(s.notna(), idx, np.nan), index=s.index).ffill()
    return pd.Series(idx, index=s.index) - last


def station_features(g: pd.DataFrame, weather: pd.DataFrame | None = None,
                     horizons=HORIZONS, with_targets: bool = True) -> pd.DataFrame:
    """g: one station, columns ts, pm25, pm10, sorted, hourly, no gaps in ts."""
    g = g.sort_values("ts").reset_index(drop=True)
    f = pd.DataFrame({"ts": g["ts"]})
    for p in POLLUTANTS:
        s = g[p]
        f[f"{p}_lag0"] = s
        for L in LAGS:
            f[f"{p}_lag{L}"] = s.shift(L)
        for w in WINDOWS:
            r = s.rolling(w, min_periods=max(1, w // 2))
            f[f"{p}_mean{w}"] = r.mean()
            f[f"{p}_std{w}"] = r.std()
        f[f"{p}_d1"] = s - s.shift(1)
        f[f"{p}_d3"] = s - s.shift(3)
        f[f"{p}_age"] = _hours_since_valid(s)
        f[f"{p}_last"] = s.ffill(limit=MAX_OBS_AGE_H)  # last reading within 3 h
    f["ratio_25_10"] = f["pm25_lag0"] / f["pm10_lag0"]
    hr = g["ts"].dt.hour
    f["hour_sin"], f["hour_cos"] = np.sin(2 * np.pi * hr / 24), np.cos(2 * np.pi * hr / 24)
    doy = g["ts"].dt.dayofyear
    f["doy_sin"], f["doy_cos"] = np.sin(2 * np.pi * doy / 365.25), np.cos(2 * np.pi * doy / 365.25)
    f["dow"] = g["ts"].dt.dayofweek

    if weather is not None:
        w = weather.set_index("ts").reindex(g["ts"]).reset_index(drop=True)
        for c in WEATHER:
            f[f"{c}_now"] = w[c].values
        f["w_rain_6h"] = w["w_rain"].rolling(6, min_periods=1).sum().values
        wd = np.deg2rad(w["w_wd"])
        f["w_u_now"], f["w_v_now"] = (-w["w_ws"] * np.sin(wd)).values, (-w["w_ws"] * np.cos(wd)).values

    rows = []
    for h in horizons:
        fh = f.copy()
        fh["horizon"] = h
        tgt_hr = (hr + h) % 24
        fh["tgt_hour_sin"], fh["tgt_hour_cos"] = np.sin(2 * np.pi * tgt_hr / 24), np.cos(2 * np.pi * tgt_hr / 24)
        for p in POLLUTANTS:
            # same clock hour as the target, one day earlier: known at t for h <= 24
            fh[f"{p}_sameh_yday"] = g[p].shift(24 - h)
            if with_targets:
                fh[f"y_{p}"] = g[p].shift(-h)
        if weather is not None:
            wt = w.shift(-h)
            for c in WEATHER:
                fh[f"{c}_tgt"] = wt[c].values
            wdt = np.deg2rad(wt["w_wd"])
            fh["w_u_tgt"], fh["w_v_tgt"] = (-wt["w_ws"] * np.sin(wdt)).values, (-wt["w_ws"] * np.cos(wdt)).values
            # rain summed over (t, t+h]
            fh["w_rain_next"] = w["w_rain"].rolling(h, min_periods=1).sum().shift(-h).values
        rows.append(fh)
    return pd.concat(rows, ignore_index=True)


def build_all(hourly: pd.DataFrame, weather: pd.DataFrame | None) -> pd.DataFrame:
    out = []
    for sid, g in hourly.groupby("station_id"):
        w = None if weather is None else weather[weather.station_id == sid]
        fs = station_features(g[["ts", "pm25", "pm10"]], w)
        fs.insert(0, "station_id", sid)
        out.append(fs)
    df = pd.concat(out, ignore_index=True)
    df["station_id"] = df["station_id"].astype("category")
    return df


def feature_columns(df: pd.DataFrame, use_tgt_weather: bool, use_station: bool = True) -> list:
    drop = {"ts", "y_pm25", "y_pm10"}
    cols = [c for c in df.columns if c not in drop and not c.startswith("y_")]
    if not use_tgt_weather:
        cols = [c for c in cols if not (c.endswith("_tgt") or c == "w_rain_next")]
    if not use_station:
        cols = [c for c in cols if c != "station_id"]
    return cols


In [ ]:
%%writefile /kaggle/working/src/airwindow/train.py
"""Baselines, gradient-boosted trees, intervals and evaluation.

Splits are chronological on issue time t; the target time t+h of the last
training row may fall a few hours into validation, so we also drop rows whose
*target* crosses a split boundary (purge), which removes any overlap.
"""
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

from .features import HORIZONS, POLLUTANTS, feature_columns

TRAIN_END = pd.Timestamp("2023-01-01 00:00")
VAL_END = pd.Timestamp("2024-01-01 00:00")
TEST_END = pd.Timestamp("2025-01-01 00:00")
QUANTILES = (0.1, 0.9)  # 80% interval


def split(df: pd.DataFrame) -> dict:
    tgt = df["ts"] + pd.to_timedelta(df["horizon"], unit="h")
    return {
        "train": df[(tgt <= TRAIN_END)],
        "val": df[(df["ts"] >= TRAIN_END) & (tgt <= VAL_END)],
        "test": df[(df["ts"] >= VAL_END) & (tgt <= TEST_END)],
    }


def usable(df: pd.DataFrame, pol: str) -> pd.DataFrame:
    """Rows where a forecast would be issued (fresh reading) and the target exists."""
    return df[df[f"{pol}_last"].notna() & df[f"y_{pol}"].notna()]


def metrics(y: np.ndarray, p: np.ndarray, p_persist: np.ndarray | None = None,
            naive_scale: float | None = None) -> dict:
    e = p - y
    out = {"n": int(len(y)), "mae": float(np.mean(np.abs(e))), "rmse": float(np.sqrt(np.mean(e ** 2))),
           "r2": float(1 - np.sum(e ** 2) / np.sum((y - y.mean()) ** 2)), "bias": float(np.mean(e))}
    if p_persist is not None:
        out["skill_vs_persistence"] = float(1 - out["mae"] / np.mean(np.abs(p_persist - y)))
    if naive_scale:
        out["mase"] = float(out["mae"] / naive_scale)
    return out


@dataclass
class TreeConfig:
    learning_rate: float = 0.05
    num_leaves: int = 127
    min_data_in_leaf: int = 200
    feature_fraction: float = 0.8
    bagging_fraction: float = 0.8
    bagging_freq: int = 1
    lambda_l2: float = 1.0
    num_boost_round: int = 4000
    early_stopping: int = 150


def fit_lgb(tr: pd.DataFrame, va: pd.DataFrame, cols: list, pol: str, cfg: TreeConfig,
            objective: str = "l2", alpha: float | None = None, log_target: bool = True):
    ytr, yva = tr[f"y_{pol}"].values, va[f"y_{pol}"].values
    if log_target:
        ytr, yva = np.log1p(ytr), np.log1p(yva)
    params = {"objective": objective, "learning_rate": cfg.learning_rate, "num_leaves": cfg.num_leaves,
              "min_data_in_leaf": cfg.min_data_in_leaf, "feature_fraction": cfg.feature_fraction,
              "bagging_fraction": cfg.bagging_fraction, "bagging_freq": cfg.bagging_freq,
              "lambda_l2": cfg.lambda_l2, "verbosity": -1, "seed": 7, "num_threads": 0}
    if alpha is not None:
        params["alpha"] = alpha
    dtr = lgb.Dataset(tr[cols], ytr, categorical_feature=[c for c in cols if c == "station_id"])
    dva = lgb.Dataset(va[cols], yva, reference=dtr)
    m = lgb.train(params, dtr, cfg.num_boost_round, valid_sets=[dva],
                  callbacks=[lgb.early_stopping(cfg.early_stopping, verbose=False), lgb.log_evaluation(0)])
    return m


def predict(m, df, cols, log_target=True):
    p = m.predict(df[cols], num_iteration=m.best_iteration)
    return np.clip(np.expm1(p) if log_target else p, 0, None)


def conformal_offset(lo, hi, y, coverage=0.8) -> float:
    """CQR: widen [lo, hi] by the quantile of non-conformity scores on validation."""
    s = np.maximum(lo - y, y - hi)
    n = len(s)
    return float(np.quantile(s, min(1.0, np.ceil((n + 1) * coverage) / n)))


def naive_scales(train: pd.DataFrame) -> dict:
    """In-sample MAE of persistence on the training split, per pollutant/horizon (MASE denominator)."""
    out = {}
    for pol in POLLUTANTS:
        u = usable(train, pol)
        for h in HORIZONS:
            uh = u[u.horizon == h]
            out[(pol, h)] = float(np.mean(np.abs(uh[f"{pol}_last"] - uh[f"y_{pol}"])))
    return out


def run_chronological(df: pd.DataFrame, out_dir: Path, cfg: TreeConfig = TreeConfig(),
                      variants=("now_weather", "tgt_weather"), log=print) -> dict:
    out_dir.mkdir(parents=True, exist_ok=True)
    sp = split(df)
    scales = naive_scales(sp["train"])
    results, preds = [], []
    for pol in POLLUTANTS:
        tr, va, te = (usable(sp[k], pol) for k in ("train", "val", "test"))
        for variant in variants:
            cols = feature_columns(df, use_tgt_weather=(variant == "tgt_weather"))
            log(f"[{pol}/{variant}] train={len(tr):,} val={len(va):,} test={len(te):,} features={len(cols)}")
            m = fit_lgb(tr, va, cols, pol, cfg)
            m.save_model(str(out_dir / f"lgb_{pol}_{variant}.txt"), num_iteration=m.best_iteration)
            q = {a: fit_lgb(tr, va, cols, pol, cfg, objective="quantile", alpha=a) for a in QUANTILES}
            for a, qm in q.items():
                qm.save_model(str(out_dir / f"lgb_{pol}_{variant}_q{int(a*100)}.txt"), num_iteration=qm.best_iteration)
            for h in HORIZONS:
                vh, th = va[va.horizon == h], te[te.horizon == h]
                lo_v, hi_v = predict(q[0.1], vh, cols), predict(q[0.9], vh, cols)
                off = conformal_offset(lo_v, hi_v, vh[f"y_{pol}"].values)
                y = th[f"y_{pol}"].values
                p = predict(m, th, cols)
                persist = th[f"{pol}_last"].values
                seas = th[f"{pol}_sameh_yday"].values
                lo = np.clip(predict(q[0.1], th, cols) - off, 0, None)
                hi = predict(q[0.9], th, cols) + off
                base = {"pollutant": pol, "horizon": h, "split": "test"}
                results.append({**base, "model": "persistence", **metrics(y, persist, persist, scales[(pol, h)])})
                ok = ~np.isnan(seas)
                results.append({**base, "model": "seasonal_naive_24h",
                                **metrics(y[ok], seas[ok], persist[ok], scales[(pol, h)]),
                                "note": "rows with a same-hour-yesterday value"})
                r = {**base, "model": f"lightgbm_{variant}", **metrics(y, p, persist, scales[(pol, h)]),
                     "interval80_coverage": float(np.mean((y >= lo) & (y <= hi))),
                     "interval80_median_width": float(np.median(hi - lo)),
                     "conformal_offset": off, "best_iteration": int(m.best_iteration)}
                results.append(r)
                log(f"  h={h} MAE persist={results[-3]['mae']:.2f} lgb={r['mae']:.2f} "
                    f"skill={r['skill_vs_persistence']:.3f} cov80={r['interval80_coverage']:.3f}")
                preds.append(pd.DataFrame({"station_id": th.station_id.values, "ts": th.ts.values,
                                           "pollutant": pol, "horizon": h, "variant": variant, "y": y,
                                           "pred": p, "lo80": lo, "hi80": hi, "persistence": persist}))
            imp = pd.Series(m.feature_importance("gain"), index=cols).sort_values(ascending=False)
            imp.head(30).to_csv(out_dir / f"importance_{pol}_{variant}.csv")
    pr = pd.concat(preds, ignore_index=True)
    pr.to_parquet(out_dir / "test_predictions_lgb.parquet", index=False)
    return {"overall": results, "per_station": per_station(pr)}


def per_station(pr: pd.DataFrame) -> list:
    rows = []
    for (sid, pol, h, var), g in pr.groupby(["station_id", "pollutant", "horizon", "variant"], observed=True):
        rows.append({"station_id": sid, "pollutant": pol, "horizon": int(h), "variant": var, "n": len(g),
                     "mae": float(np.mean(np.abs(g.pred - g.y))),
                     "mae_persistence": float(np.mean(np.abs(g.persistence - g.y))),
                     "rmse": float(np.sqrt(np.mean((g.pred - g.y) ** 2)))})
    return rows


def run_leave_station_out(df: pd.DataFrame, holdout: list, cfg: TreeConfig = TreeConfig(),
                          variant: str = "tgt_weather", log=print) -> list:
    """Train without the held-out station (and without station_id), test on it in 2024."""
    sp = split(df)
    cols = feature_columns(df, use_tgt_weather=(variant == "tgt_weather"), use_station=False)
    rows = []
    for sid in holdout:
        for pol in POLLUTANTS:
            tr = usable(sp["train"][sp["train"].station_id != sid], pol)
            va = usable(sp["val"][sp["val"].station_id != sid], pol)
            te = usable(sp["test"][sp["test"].station_id == sid], pol)
            if len(te) < 500:
                log(f"skip {sid}/{pol}: only {len(te)} test rows")
                continue
            m = fit_lgb(tr, va, cols, pol, cfg)
            for h in HORIZONS:
                th = te[te.horizon == h]
                y, p, persist = th[f"y_{pol}"].values, predict(m, th, cols), th[f"{pol}_last"].values
                rows.append({"held_out_station": sid, "pollutant": pol, "horizon": h, "variant": variant,
                             **metrics(y, p, persist)})
                log(f"LOSO {sid} {pol} h={h} MAE={rows[-1]['mae']:.2f} skill={rows[-1]['skill_vs_persistence']:.3f}")
    return rows


def save_json(obj, path: Path) -> None:
    path.write_text(json.dumps(obj, indent=1, default=str))


In [ ]:
%%writefile /kaggle/working/src/airwindow/deep.py
"""GRU forecaster: past 48 h sequence -> PM2.5/PM10 at +1/+3/+6 h in one pass.

Inputs per hour: log1p PM2.5/PM10 (scaled), their missing masks, weather at that
hour, hour-of-day. Optional decoder vector: weather at each target hour (the
"forecast weather" variant). The scaler is fitted on the training split only.
Windows are gathered on the device from one contiguous array, so nothing is
materialised per sample. Missing targets are masked out of the loss.
"""
from __future__ import annotations

import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from .features import HORIZONS, MAX_OBS_AGE_H, POLLUTANTS, WEATHER
from .train import TEST_END, TRAIN_END, VAL_END, metrics

SEQ = 48
SEQ_FEATS = ["pm25_l", "pm10_l", "pm25_m", "pm10_m", *WEATHER, "hour_sin", "hour_cos"]
TGT_W = ["w_temp", "w_rh", "w_ws", "w_rain", "w_blh"]


class GRUForecaster(nn.Module):
    def __init__(self, n_in, n_stations, n_tgt_w, hidden=128, layers=2, emb=8, dropout=0.1):
        super().__init__()
        self.emb = nn.Embedding(n_stations, emb)
        self.gru = nn.GRU(n_in, hidden, layers, batch_first=True, dropout=dropout)
        self.head = nn.Sequential(nn.Linear(hidden + emb + n_tgt_w, 128), nn.GELU(), nn.Dropout(dropout),
                                  nn.Linear(128, len(POLLUTANTS) * len(HORIZONS)))

    def forward(self, x, sid, tw):
        _, h = self.gru(x)
        z = torch.cat([h[-1], self.emb(sid), tw], dim=1)
        return self.head(z)


def prepare(hourly: pd.DataFrame, weather: pd.DataFrame, use_tgt_weather: bool):
    """Concatenate all stations into one array; return arrays + sample index table."""
    st_ids = sorted(hourly.station_id.unique())
    sid_map = {s: i for i, s in enumerate(st_ids)}
    parts, samples, off = [], [], 0
    for sid in st_ids:
        g = hourly[hourly.station_id == sid].sort_values("ts").reset_index(drop=True)
        w = weather[weather.station_id == sid].set_index("ts").reindex(g.ts).reset_index(drop=True)
        a = pd.DataFrame({"ts": g.ts})
        for p in POLLUTANTS:
            a[f"{p}_l"] = np.log1p(g[p])
            a[f"{p}_m"] = g[p].isna().astype(float)
            a[f"y_{p}"] = g[p]
        for c in WEATHER:
            a[c] = w[c].values
        a["hour_sin"], a["hour_cos"] = np.sin(2 * np.pi * g.ts.dt.hour / 24), np.cos(2 * np.pi * g.ts.dt.hour / 24)
        n = len(a)
        # sample = issue index i with i >= SEQ-1 and i + max(h) < n
        i = np.arange(SEQ - 1, n - max(HORIZONS))
        fresh = {}
        for p in POLLUTANTS:
            last_valid = pd.Series(np.where(g[p].notna(), np.arange(n), np.nan)).ffill().values
            fresh[p] = (np.arange(n) - last_valid) <= MAX_OBS_AGE_H
        s = pd.DataFrame({"row": i + off, "local": i, "ts": a.ts.values[i], "sid": sid_map[sid],
                          "fresh_pm25": fresh["pm25"][i], "fresh_pm10": fresh["pm10"][i]})
        samples.append(s)
        a["station_id"] = sid
        parts.append(a)
        off += n
    arr = pd.concat(parts, ignore_index=True)
    smp = pd.concat(samples, ignore_index=True)
    return arr, smp, st_ids


class Data:
    def __init__(self, arr: pd.DataFrame, device, use_tgt_weather):
        tr_hours = arr[arr.ts < TRAIN_END]  # scaler sees training-period hours only
        self.mu = tr_hours[SEQ_FEATS].mean().values
        self.sd = tr_hours[SEQ_FEATS].std().replace(0, 1).values
        x = (arr[SEQ_FEATS].values - self.mu) / self.sd
        self.x = torch.tensor(np.nan_to_num(x, nan=0.0), dtype=torch.float32, device=device)
        y = np.stack([np.log1p(arr[f"y_{p}"].values) for p in POLLUTANTS], 1)
        self.y = torch.tensor(y, dtype=torch.float32, device=device)  # NaN where missing
        self.use_tw = use_tgt_weather
        tw_idx = [SEQ_FEATS.index(c) for c in TGT_W]
        self.tw_cols = torch.tensor(tw_idx, device=device)
        self.device = device
        self.hz = torch.tensor(HORIZONS, device=device)
        self.back = torch.arange(SEQ - 1, -1, -1, device=device)

    def batch(self, rows: np.ndarray, sids: np.ndarray):
        r = torch.as_tensor(rows, device=self.device)
        xb = self.x[r[:, None] - self.back[None, :]]            # B x SEQ x F
        tgt = r[:, None] + self.hz[None, :]                       # B x H
        yb = self.y[tgt]                                          # B x H x P
        yb = yb.permute(0, 2, 1).reshape(len(rows), -1)           # B x (P*H), pol-major
        if self.use_tw:
            tw = self.x[tgt][:, :, self.tw_cols].reshape(len(rows), -1)
        else:
            tw = torch.zeros(len(rows), 0, device=self.device)
        return xb, torch.as_tensor(sids, device=self.device), tw, yb


def train_gru(hourly, weather, use_tgt_weather=True, epochs=12, batch=2048, lr=2e-3, log=print, seed=7):
    torch.manual_seed(seed)
    np.random.seed(seed)
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    arr, smp, st_ids = prepare(hourly, weather, use_tgt_weather)
    tgt_end = smp.ts + pd.Timedelta(hours=max(HORIZONS))
    fresh_any = smp.fresh_pm25 | smp.fresh_pm10
    m_tr = (tgt_end <= TRAIN_END) & fresh_any
    m_va = (smp.ts >= TRAIN_END) & (tgt_end <= VAL_END) & fresh_any
    m_te = (smp.ts >= VAL_END) & (tgt_end <= TEST_END)
    d = Data(arr, dev, use_tgt_weather)
    n_tw = len(TGT_W) * len(HORIZONS) if use_tgt_weather else 0
    model = GRUForecaster(len(SEQ_FEATS), len(st_ids), n_tw).to(dev)
    with torch.no_grad():  # start at the training mean (log space) instead of 0
        y_tr = d.y[torch.as_tensor(smp.row.values[m_tr.values], device=dev)]
        model.head[-1].bias.copy_(torch.nanmean(y_tr, 0).repeat_interleave(len(HORIZONS)))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    tr_rows, tr_sid = smp.row.values[m_tr], smp.sid.values[m_tr]
    va_rows, va_sid = smp.row.values[m_va], smp.sid.values[m_va]
    steps = epochs * int(np.ceil(len(tr_rows) / batch))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps)
    lossf = nn.HuberLoss(reduction="none", delta=0.5)

    def masked(pred, yb):
        m = ~torch.isnan(yb)
        return (lossf(pred, torch.nan_to_num(yb)) * m).sum() / m.sum().clamp(min=1)

    best, best_state, hist = 1e9, None, []
    for ep in range(epochs):
        model.train()
        t0 = time.time()
        perm = np.random.permutation(len(tr_rows))
        tot = 0.0
        for k in range(0, len(perm), batch):
            j = perm[k:k + batch]
            xb, sb, tw, yb = d.batch(tr_rows[j], tr_sid[j])
            loss = masked(model(xb, sb, tw), yb)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            tot += loss.item() * len(j)
        vl = evaluate_loss(model, d, va_rows, va_sid, masked)
        hist.append({"epoch": ep + 1, "train_loss": tot / len(perm), "val_loss": vl, "sec": time.time() - t0})
        log(f"epoch {ep+1} train={tot/len(perm):.4f} val={vl:.4f} ({time.time()-t0:.0f}s)")
        if vl < best:
            best, best_state = vl, {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, d, arr, smp, m_te.values, st_ids, hist


@torch.no_grad()
def evaluate_loss(model, d, rows, sids, masked, batch=8192):
    model.eval()
    tot, n = 0.0, 0
    for k in range(0, len(rows), batch):
        xb, sb, tw, yb = d.batch(rows[k:k + batch], sids[k:k + batch])
        tot += masked(model(xb, sb, tw), yb).item() * len(xb)
        n += len(xb)
    return tot / max(n, 1)


@torch.no_grad()
def predict_test(model, d, arr, smp, m_te, batch=8192):
    model.eval()
    rows, sids = smp.row.values[m_te], smp.sid.values[m_te]
    out = []
    for k in range(0, len(rows), batch):
        xb, sb, tw, _ = d.batch(rows[k:k + batch], sids[k:k + batch])
        out.append(torch.expm1(model(xb, sb, tw)).clamp(min=0).cpu().numpy())
    P = np.concatenate(out)
    recs = []
    for pi, pol in enumerate(POLLUTANTS):
        for hi, h in enumerate(HORIZONS):
            recs.append(pd.DataFrame({"row": rows, "pollutant": pol, "horizon": h,
                                      "pred": P[:, pi * len(HORIZONS) + hi]}))
    return pd.concat(recs, ignore_index=True)


def score_against(pred_gru: pd.DataFrame, smp: pd.DataFrame, arr: pd.DataFrame, lgb_test: pd.DataFrame,
                  variant: str) -> list:
    """Score GRU on exactly the rows the tree models were scored on (same station, ts, horizon)."""
    key = smp[["row", "ts"]].assign(station_id=arr.station_id.values[smp.row.values])
    g = pred_gru.merge(key, on="row")
    ref = lgb_test[lgb_test.variant == variant][["station_id", "ts", "pollutant", "horizon", "y", "persistence"]]
    m = ref.merge(g[["station_id", "ts", "pollutant", "horizon", "pred"]],
                  on=["station_id", "ts", "pollutant", "horizon"], how="inner")
    res = []
    for (pol, h), x in m.groupby(["pollutant", "horizon"]):
        res.append({"pollutant": pol, "horizon": int(h), "split": "test", "model": f"gru_{variant}",
                    **metrics(x.y.values, x.pred.values, x.persistence.values)})
    return res, m


In [ ]:
%%writefile /kaggle/working/src/airwindow/run_experiment.py
"""End-to-end experiment: features -> baselines + LightGBM (+intervals) -> LOSO -> GRU -> report.

    python -m airwindow.run_experiment --data data/processed --out ml/artifacts [--quick] [--no-gru]
"""
from __future__ import annotations

import argparse
import json
import platform
import time
from pathlib import Path

import numpy as np
import pandas as pd

from . import train as T
from .features import HORIZONS, POLLUTANTS, build_all, feature_columns

LOSO_STATIONS = ["dhk_darussalam", "ctg_agrabad", "sylhet", "rajshahi", "khulna"]


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", type=Path, default=Path("data/processed"))
    ap.add_argument("--out", type=Path, default=Path("ml/artifacts"))
    ap.add_argument("--quick", action="store_true", help="2 stations, few rounds: smoke test only")
    ap.add_argument("--no-gru", action="store_true")
    ap.add_argument("--gru-epochs", type=int, default=12)
    a = ap.parse_args()
    a.out.mkdir(parents=True, exist_ok=True)
    logf = open(a.out / "train_log.txt", "a")

    def log(msg):
        print(msg, flush=True)
        logf.write(msg + "\n")
        logf.flush()

    t0 = time.time()
    hourly = pd.read_parquet(a.data / "cams_bd_hourly.parquet", columns=["station_id", "ts", "pm25", "pm10"])
    weather = pd.read_parquet(a.data / "era5_station_weather.parquet")
    for d in (hourly, weather):  # upload copies store these as categoricals
        d["station_id"] = d["station_id"].astype(str)
    cfg = T.TreeConfig()
    loso = LOSO_STATIONS
    if a.quick:
        keep = ["dhk_darussalam", "khulna"]
        hourly, weather = hourly[hourly.station_id.isin(keep)], weather[weather.station_id.isin(keep)]
        cfg = T.TreeConfig(num_boost_round=60, early_stopping=20)
        loso = []
    feats = build_all(hourly, weather)
    log(f"feature table {feats.shape} in {time.time()-t0:.0f}s")

    res = T.run_chronological(feats, a.out, cfg, log=log)
    res["leave_station_out"] = T.run_leave_station_out(feats, loso, cfg, log=log) if loso else []

    if not a.no_gru:
        import torch
        from . import deep
        lgb_test = pd.read_parquet(a.out / "test_predictions_lgb.parquet")
        res["gru"], res["gru_history"] = [], {}
        for variant, use_tw in (("tgt_weather", True), ("now_weather", False)):
            model, d, arr, smp, m_te, st_ids, hist = deep.train_gru(
                hourly, weather, use_tgt_weather=use_tw, epochs=2 if a.quick else a.gru_epochs, log=log)
            pg = deep.predict_test(model, d, arr, smp, m_te)
            scores, merged = deep.score_against(pg, smp, arr, lgb_test, variant)
            for s in scores:
                log(f"GRU {variant} {s['pollutant']} h={s['horizon']} MAE={s['mae']:.2f} skill={s['skill_vs_persistence']:.3f}")
            res["gru"] += scores
            res["gru_history"][variant] = hist
            torch.save({"state_dict": model.state_dict(), "mu": d.mu.tolist(), "sd": d.sd.tolist(),
                        "stations": st_ids, "use_tgt_weather": use_tw}, a.out / f"gru_{variant}.pt")
            merged.to_parquet(a.out / f"test_predictions_gru_{variant}.parquet", index=False)
        res["gru_device"] = "cuda" if torch.cuda.is_available() else "cpu"
        if torch.cuda.is_available():
            res["gpu_name"] = torch.cuda.get_device_name(0)

    sp = T.split(feats)
    res["meta"] = {
        "dataset": "CAMS-BD (doi:10.17632/hwmzyh5w4w.1), CC BY 4.0; weather: Open-Meteo ERA5 archive",
        "splits": {"train": f"targets <= {T.TRAIN_END}", "val": f"{T.TRAIN_END} .. {T.VAL_END}",
                   "test": f"{T.VAL_END} .. {T.TEST_END}"},
        "rows": {k: int(len(v)) for k, v in sp.items()},
        "horizons_h": list(HORIZONS), "pollutants": list(POLLUTANTS),
        "features_now_weather": feature_columns(feats, False), "features_tgt_weather": feature_columns(feats, True),
        "interval": "LightGBM q10/q90 widened by split-conformal offset fitted on validation (CQR), nominal 80%",
        "tgt_weather_caveat": "Uses ERA5 reanalysis at the target hour as a stand-in for the weather forecast "
                              "the app fetches live; real forecasts are less accurate, so treat as optimistic.",
        "quick_mode": bool(a.quick), "python": platform.python_version(), "runtime_s": round(time.time() - t0),
    }
    T.save_json(res, a.out / "metrics.json")
    log(f"done in {time.time()-t0:.0f}s")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/src/airwindow/build_dataset.py
"""Build the processed CAMS-BD dataset and its data-quality report.

    python -m airwindow.build_dataset --raw data/raw/CAMS-BD --out data/processed [--weather]

Outputs:
    cams_bd_hourly.parquet   cleaned station-hour table on a complete hourly grid
    era5_station_weather.parquet  (with --weather) Open-Meteo ERA5 per station-hour
    data_quality.json / DATA_QUALITY.md
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import pandas as pd

from .cleaning import clean, to_hourly_grid
from .parsing import CANONICAL, parse_all
from .stations import STATIONS


def quality_markdown(q: dict) -> str:
    L = ["# CAMS-BD data-quality report", "",
         f"Source rows read: **{q['parse']['rows_in']:,}** · station-hours kept: **{q['rows_clean']:,}** · "
         f"with PM2.5: **{q['pm25_nonnull']:,}** · with PM10: **{q['pm10_nonnull']:,}**", "",
         "## Rows dropped while parsing", "", "| Reason | Rows |", "|---|---:|"]
    L += [f"| {k} | {v:,} |" for k, v in q["parse"]["dropped"].items()]
    L += ["", "## Timestamp repairs (2012-2021 workbook)", "", "| Repair | Rows |", "|---|---:|"]
    L += [f"| {k} | {v:,} |" for k, v in q["parse"]["date_fixes"].items()]
    L += ["", "## Station coverage", "", "| Station | First | Last | Hours on grid | PM2.5 present | PM10 present |",
          "|---|---|---|---:|---:|---:|"]
    for s in q["coverage"]:
        L.append(f"| {s['station_id']} | {s['first']} | {s['last']} | {s['hours']:,} | "
                 f"{s['pm25_frac']:.0%} | {s['pm10_frac']:.0%} |")
    L += ["", "## Values nulled by physical-range checks", "", "| Column | Values |", "|---|---:|"]
    L += [f"| {k} | {v:,} |" for k, v in q["clean"]["range_nulled"].items()]
    L += ["", f"Stuck-sensor PM values nulled (≥6 identical consecutive hours): "
          f"PM2.5 {q['clean']['stuck_nulled']['pm25']:,}, PM10 {q['clean']['stuck_nulled']['pm10']:,}.", "",
          "## Station-year columns quarantined (>20% of values physically impossible)", "",
          "| Station | Year | Column | Bad fraction | Values nulled |", "|---|---:|---|---:|---:|"]
    L += [f"| {r['station_id']} | {r['year']} | {r['column']} | {r['bad_frac']:.0%} | {r['n_nulled']:,} |"
          for r in q["clean"]["quarantined"]]
    L += ["", f"Duplicates on (station, hour): {json.dumps(q['clean']['duplicates'])}", "",
          f"PM2.5 > PM10 (physically inconsistent, kept but flagged): {q['pm25_gt_pm10']:,} station-hours.", "",
          "## Unit labels that differ from the expected unit", "", "| Source | Station | Column | Unit as written |",
          "|---|---|---|---|"]
    L += [f"| {u['source']} | {u['station']} | {u['column']} | {u['unit']} |" for u in q["unit_issues"]]
    L += ["", "## Notes", ""] + [f"- {n}" for n in q["parse"]["notes"]]
    return "\n".join(L) + "\n"


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--raw", type=Path, default=Path("data/raw/CAMS-BD"))
    ap.add_argument("--out", type=Path, default=Path("data/processed"))
    ap.add_argument("--weather", action="store_true", help="also download ERA5 weather from Open-Meteo")
    a = ap.parse_args()
    a.out.mkdir(parents=True, exist_ok=True)

    raw, audit = parse_all(a.raw)
    df, rep = clean(raw)
    grid = to_hourly_grid(df)
    grid = grid[["station_id", "ts", *CANONICAL, "source_file"]]
    grid.to_parquet(a.out / "cams_bd_hourly.parquet", index=False)

    cov = []
    for sid, g in grid.groupby("station_id"):
        cov.append({"station_id": sid, "first": str(g.ts.min()), "last": str(g.ts.max()), "hours": len(g),
                    "pm25_frac": float(g.pm25.notna().mean()), "pm10_frac": float(g.pm10.notna().mean())})
    uniq_units = {(u["source"], u["station"], u["column"], u["unit"]): u for u in audit.unit_issues}
    q = {"parse": {**audit.to_dict(), "unit_issues": None}, "clean": rep, "coverage": cov,
         "rows_clean": int(len(df)), "grid_rows": int(len(grid)),
         "pm25_nonnull": int(df.pm25.notna().sum()), "pm10_nonnull": int(df.pm10.notna().sum()),
         "pm25_gt_pm10": int((df.pm25 > df.pm10).sum()), "unit_issues": list(uniq_units.values()),
         "timezone": "Asia/Dhaka (UTC+6), naive; ts = end of averaging hour",
         "stations": STATIONS}
    (a.out / "data_quality.json").write_text(json.dumps(q, indent=1, default=str))
    (a.out / "DATA_QUALITY.md").write_text(quality_markdown(q))
    print(f"wrote {len(grid):,} grid rows; PM2.5 present {q['pm25_nonnull']:,}")

    if a.weather:
        fetch_station_weather(grid, a.out)


def fetch_station_weather(grid: pd.DataFrame, out: Path, pause_s: float = 20.0) -> None:
    """ERA5 per station, cached one file per station (Open-Meteo rate-limits long ranges)."""
    import time
    from .weather import fetch_archive
    cache = out / "weather_cache"
    cache.mkdir(exist_ok=True)
    frames = []
    for sid, g in grid.groupby("station_id"):
        f = cache / f"{sid}.parquet"
        if not f.exists():
            st = STATIONS[sid]
            w = fetch_archive(st["lat"], st["lon"], str(g.ts.min().date()), str(g.ts.max().date()))
            w.insert(0, "station_id", sid)
            w.to_parquet(f, index=False)
            print("weather", sid, len(w), flush=True)
            time.sleep(pause_s)
        frames.append(pd.read_parquet(f))
    pd.concat(frames, ignore_index=True).to_parquet(out / "era5_station_weather.parquet", index=False)

if __name__ == "__main__":
    main()


In [ ]:
import glob, os, subprocess, sys, json
hits = glob.glob('/kaggle/input/**/cams_bd_hourly.parquet', recursive=True)
assert hits, 'Attach the AirWindow processed dataset (cams_bd_hourly.parquet)'
DATA = os.path.dirname(hits[0]); print('data dir:', DATA, os.listdir(DATA))
import torch, lightgbm; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '', '| lightgbm', lightgbm.__version__)
!nvidia-smi -L || true

In [ ]:
%cd /kaggle/working/src
!python -m pytest --version >/dev/null 2>&1 || pip -q install pytest
!python -m airwindow.run_experiment --data {DATA} --out /kaggle/working/artifacts --no-gru

In [ ]:
import pandas as pd, json
M = json.load(open('/kaggle/working/artifacts/metrics.json'))
ov = pd.DataFrame(M['overall'] + M.get('gru', []))
cols = ['pollutant','horizon','model','n','mae','rmse','r2','skill_vs_persistence','interval80_coverage','interval80_median_width']
pd.set_option('display.width', 200)
display(ov[[c for c in cols if c in ov]].sort_values(['pollutant','horizon','mae']).round(3))
display(pd.DataFrame(M['leave_station_out']).round(3))

In [ ]:
import matplotlib.pyplot as plt
pr = pd.read_parquet('/kaggle/working/artifacts/test_predictions_lgb.parquet')
s = pr[(pr.station_id=='dhk_darussalam')&(pr.pollutant=='pm25')&(pr.horizon==3)&(pr.variant=='tgt_weather')]
s = s[(s.ts>='2024-01-10')&(s.ts<'2024-01-24')].sort_values('ts')
fig, ax = plt.subplots(figsize=(13,4))
ax.fill_between(s.ts, s.lo80, s.hi80, alpha=.25, label='80% interval')
ax.plot(s.ts, s.y, 'k.', ms=3, label='observed (t+3h)'); ax.plot(s.ts, s.pred, label='LightGBM forecast')
ax.set_ylabel('PM2.5 µg/m³'); ax.legend(); ax.set_title('Dhaka Darussalam, 3 h ahead, test period (Jan 2024)')
plt.savefig('/kaggle/working/artifacts/example_forecast.png', dpi=120, bbox_inches='tight'); plt.show()

In [ ]:
!cd /kaggle/working && zip -qr airwindow_artifacts.zip artifacts && ls -la /kaggle/working /kaggle/working/artifacts